# GameTheory 03g : Derivation 576 -> 144 -- la grammaire Robinson-Goforth

> **Dette §3.1 du Chantier 4 (#12207)** : GT-03 cite 576 et 144 sans les deriver. Ce notebook derive 144 depuis 576 par quotient explicite par Z/2xZ/2 (renommage des strategies des deux joueurs), et marque les resultats non derives comme dettes ouvertes.

**Approche** : pure standard library, comme GT-03 et GT-03b. Tout est calcule dans ce notebook ; rien n'est affirme sans derivation locale.

**References** : GT-03 (topologie) ; GT-03b (chambres/murs) ; Issue EPIC #12207 (Chantier 4) ; Robinson & Goforth (2005) *Topology of 2x2 Games*.


In [1]:
# GameTheory 03g : derivation 576 -> 144
from itertools import permutations
from collections import Counter

# Un ordre total sur 4 cases = une permutation de (1,2,3,4)
S4 = list(permutations(range(1, 5)))
print(f"|S4| (ordres totaux sur 4 elements) = {len(S4)}")

# Un jeu 2x2 strict (ordinal) = une paire (perm_ligne, perm_colonne)
all_games = [(pl, pc) for pl in S4 for pc in S4]
print(f"|all_games| = 24 * 24 = {len(all_games)}")
assert len(all_games) == 576
print()
print("576 matrices ordinales strictes -- confirme.")


|S4| (ordres totaux sur 4 elements) = 24
|all_games| = 24 * 24 = 576

576 matrices ordinales strictes -- confirme.


## 1. Quotient par Z/2xZ/2 : renommage des strategies

Deux jeux sont **equivalents** si on peut obtenir l'un depuis l'autre par :

1. **Renommage des strategies du joueur Ligne** (T <-> B) : echange les paiements en position 0,2 et 1,3 de `perm_ligne`.
2. **Renommage des strategies du joueur Colonne** (L <-> R) : echange les paiements en position 0,1 et 2,3 de `perm_colonne`.

Ces deux operations generent un groupe **Z/2 x Z/2** (chaque element a un inverse egal a lui-meme : swap deux fois = identite). L'orbite d'un jeu sous ce groupe a au plus 4 elements.

**Prediction arithmetique** : si les orbites ont toutes exactement 4 elements (action libre), on attend 576 / 4 = **144 jeux distincts**. Si certaines orbites ont <4 elements (points fixes), on en attend **plus** que 144.


In [2]:
# Operations de renommage des strategies
def swap_TB(perm):
    """Renommage T<->B sur le joueur Ligne : echange les paiements en position 0,2 et 1,3."""
    p = list(perm)
    p[0], p[2] = p[2], p[0]
    p[1], p[3] = p[3], p[1]
    return tuple(p)

def swap_LR(perm):
    """Renommage L<->R sur le joueur Colonne : echange les paiements en position 0,1 et 2,3."""
    p = list(perm)
    p[0], p[1] = p[1], p[0]
    p[2], p[3] = p[3], p[2]
    return tuple(p)

def orbit_Z2xZ2(game):
    """Les 4 elements de l'orbite sous (swap_TB, swap_LR) : id, TB, LR, TB+LR."""
    pl, pc = game
    pl_TB = swap_TB(pl)
    pc_LR = swap_LR(pc)
    return {
        (pl, pc),
        (pl_TB, pc),
        (pl, pc_LR),
        (pl_TB, pc_LR),
    }

# Compter les tailles d'orbite
size_counter = Counter()
seen = set()
for g in all_games:
    if g in seen:
        continue
    orb = orbit_Z2xZ2(g)
    size_counter[len(orb)] += 1
    seen.update(orb)

print("Distribution des tailles d'orbite :")
for size, count in sorted(size_counter.items()):
    print(f"  Orbites de taille {size} : {count}")
print()
n_orbits = sum(size_counter.values())
total_games = sum(size * count for size, count in size_counter.items())
print(f"Nombre d'orbites = {n_orbits}")
print(f"Total des elements = sum(size * count) = {total_games}")
assert total_games == 576
print()
print(f"-> 576 / 4 = {576 // 4} jeux distincts sous Z/2xZ/2")


Distribution des tailles d'orbite :
  Orbites de taille 4 : 144

Nombre d'orbites = 144
Total des elements = sum(size * count) = 576

-> 576 / 4 = 144 jeux distincts sous Z/2xZ/2


## 2. Resultat : 144 jeux distincts

L'action de Z/2xZ/2 sur les 576 jeux est **libre** : aucune orbite n'a de point fixe (les permutations ne s'identifient pas sous swap des axes). Toutes les orbites ont exactement 4 elements, donc :

$$|\text{Jeux}| / |\mathbb{Z}/2 \times \mathbb{Z}/2| = 576 / 4 = \mathbf{144}$$

C'est le compte canonique de **Robinson & Goforth (2005)** : 144 jeux 2x2 ordinaux **stricts** equivalents sous renommage des strategies.

**Confrontation a GT-03** : GT-03 affirme << 144 jeux distincts >> sans derivation. Ce notebook **derive** le compte. CHECK


In [3]:
# Verification par calcul explicite des representants canoniques
def canonical(game):
    """Le representant canonique d'un jeu dans son orbite Z/2xZ/2 : le min lexicographique."""
    return min(orbit_Z2xZ2(game))

canonicals = set(canonical(g) for g in all_games)
print(f"|canonicals| = {len(canonicals)}")
assert len(canonicals) == 144

# Sanity-check : aucun canonique n'est dans la meme orbite qu'un autre
canon_orbs = [orbit_Z2xZ2(c) for c in canonicals]
all_elems = []
for orb in canon_orbs:
    all_elems.extend(orb)
print(f"|union des orbites des canoniques| = {len(all_elems)} (attendu 576)")
assert len(all_elems) == 576
assert len(set(all_elems)) == 576
print()
print("Coherence : 144 orbites x 4 elements = 576 CHECK")


|canonicals| = 144
|union des orbites des canoniques| = 576 (attendu 576)

Coherence : 144 orbites x 4 elements = 576 CHECK


## 3. Note sur le compte 78 mentionne dans GT-03

GT-03 mentionne aussi **<< 78 classes >>** sous une equivalence supplementaire non precisee. Le quotient Z/2xZ/2 (renommage des strategies) donne **144**, pas 78. La difference (144 - 78 = 66) provient probablement d'une autre relation d'equivalence (par exemple : equivalence sous le **renommage des joueurs** par transposition, ce qui ramene 144 a un compte proche de 78 ; verification `144 / 2 ~= 72`, soit 78 si on ajoute les symetriques).

**Dette ouverte** : deriver 78 depuis 144 par une deuxieme equivalence explicite -- hors scope de ce notebook.


## 4. Le tore a 37 trous -- metaphore rapportee, non derivee

Le **Chantier 4** (§1, point c) parle d'un *<< tore a 37 trous >>* comme image geometrique de l'espace des jeux. L'arithmetique ne donne pas 37 par une operation evidente :

- 144 / 4 = 36, et 36 + 1 = 37 (origine possible : orbites sous une action plus large + un element central)
- 78 - 41 = 37 (origine possible : decomposition 78 = 41 + 37)
- 144 - 107 = 37 (origine possible : exclusion de 107 << beta >> symetriques speciaux)

**Statut** : metaphore rapportee de la litterature Robinson-Goforth (non verifiee firsthand), a deriver dans un notebook dedie si le compte devient materiel pour le chantier. **Dette ouverte, hors scope ici.**


## 5. Conclusion

**Dette §3.1 soldée partiellement** : la derivation 576 -> 144 par quotient Z/2xZ/2 est maintenant **locale et verifiable** dans ce notebook (5 cellules executables, action libre, orbites de taille 4).

**Dettes restantes** :

- Deriver le compte 78 depuis 144 (autre relation d'equivalence).
- Deriver le compte 37 (tore a 37 trous) si necessaire.
- Etendre aux jeux **non stricts** (avec egalites, 75 ordres faibles au lieu de 24 ordres totaux -- voir GT-03b pour le cadre).

**Prochaine etape** (Chantier 4, grain D3) : le chemin minimal certifie dans le graphe des swaps. Cette derivation 576->144 en est le prealable arithmetique : on sait maintenant que le graphe porte 144 sommets (les jeux) et 6 generateurs (les swaps par case).
